# Workflow Orchestration and Agentic RAG

Yesterday's agent chose its own steps. Most production systems should not:
when you know the steps, **encode them** — explicit control flow is
testable, observable, and cheap. This session builds an orchestrated
pipeline whose only "agency" is one well-placed decision: *is my retrieval
good enough, or should I rewrite the query and try again?* That pattern is
agentic RAG.

## Steps as functions over shared state

Each step reads and writes one `state` dict. That convention — not any
framework — is what makes steps composable and traceable.

In [1]:
# The knowledge base for every lab this week: support documents for Atlas
# Cycles, a fictional e-bike maker. Small enough to read, real enough to
# retrieve against.
CORPUS = {
    "battery-care": (
        "Atlas S2 battery care. Charge the battery to 80 percent for daily "
        "use and only to 100 percent before a long ride. Store between 10 "
        "and 25 degrees Celsius. A full recharge takes 4.5 hours from empty."
    ),
    "warranty": (
        "Atlas warranty policy. The frame is covered for 5 years. The "
        "battery and motor are covered for 2 years or 15,000 km, whichever "
        "comes first. Wear parts such as brake pads and tires are excluded."
    ),
    "range": (
        "Atlas S2 range guide. Expect 90 to 110 km in Eco mode, 60 to 75 km "
        "in Trail mode, and 40 to 55 km in Boost mode. Headwind, cargo "
        "weight, and cold weather reduce range by up to 30 percent."
    ),
    "error-codes": (
        "Atlas display error codes. E01 means a motor sensor fault: restart "
        "the system. E04 means battery communication lost: reseat the "
        "battery. E09 means brake cutoff engaged: check the brake levers."
    ),
    "first-service": (
        "First service. Book the complimentary first service after 300 km "
        "or 3 months. Spoke tension, brake bedding, and firmware updates "
        "are included at no charge."
    ),
}

# A deterministic embedding: character trigram counts. No model, no network,
# yet it captures enough word-shape overlap to demonstrate the geometry that
# real embedding models learn.
from collections import Counter
import math

def embed(text):
    t = " " + "".join(c.lower() if c.isalnum() else " " for c in text) + " "
    return Counter(t[i:i+3] for i in range(len(t) - 2) if t[i:i+3].strip())

def cosine(a, b):
    dot = sum(a[k] * b[k] for k in a.keys() & b.keys())
    na = math.sqrt(sum(v * v for v in a.values()))
    nb = math.sqrt(sum(v * v for v in b.values()))
    return dot / (na * nb) if na and nb else 0.0

DOC_VECTORS = {name: embed(doc) for name, doc in CORPUS.items()}

def retrieve(state):
    q = embed(state["query"])
    ranked = sorted(((cosine(q, v), n) for n, v in DOC_VECTORS.items()),
                    reverse=True)
    state["hits"] = ranked[:2]
    return state

def grade(state):
    # Self-check: is the best hit actually about the question?
    best_score = state["hits"][0][0]
    state["grade"] = "good" if best_score >= 0.30 else "weak"
    return state

def rewrite_query(state):
    # One deterministic expansion: add domain synonyms for common phrasings.
    synonyms = {"checkup": "first service", "screen": "display error",
                "broken": "error code", "how far": "range km"}
    q = state["query"].lower()
    for informal, formal in synonyms.items():
        if informal in q:
            q = q.replace(informal, formal)
    state["query"] = q
    state["rewritten"] = True
    return state

def draft(state):
    name = state["hits"][0][1]
    state["answer"] = f"{CORPUS[name][:90]}... [source: {name}]"
    return state

print("steps defined: retrieve, grade, rewrite_query, draft")

steps defined: retrieve, grade, rewrite_query, draft


## The orchestrator

A pipeline with one branch and one retry. Every transition prints — in
production these lines are spans in your tracing system, and the branch
decision is a metric you alert on.

In [2]:
def answer_with_agentic_rag(query):
    state = {"query": query, "rewritten": False}
    print(f"query: {query!r}")
    state = retrieve(state)
    state = grade(state)
    print(f"  retrieve -> top={state['hits'][0][1]} "
          f"score={state['hits'][0][0]:.3f} grade={state['grade']}")
    if state["grade"] == "weak" and not state["rewritten"]:
        state = rewrite_query(state)
        print(f"  rewrite  -> {state['query']!r}")
        state = retrieve(state)
        state = grade(state)
        print(f"  retrieve -> top={state['hits'][0][1]} "
          f"score={state['hits'][0][0]:.3f} grade={state['grade']}")
    state = draft(state)
    print(f"  draft    -> {state['answer'][:70]}...")
    return state["answer"]

# A query that retrieves cleanly on the first pass:
answer_with_agentic_rag("what does error code E09 mean")
print()
# And one whose informal phrasing needs the rewrite loop:
answer_with_agentic_rag("when is the bike's first checkup")

query: 'what does error code E09 mean'
  retrieve -> top=error-codes score=0.311 grade=good
  draft    -> Atlas display error codes. E01 means a motor sensor fault: restart the...

query: "when is the bike's first checkup"
  retrieve -> top=first-service score=0.237 grade=weak
  rewrite  -> "when is the bike's first first service"
  retrieve -> top=first-service score=0.468 grade=good
  draft    -> First service. Book the complimentary first service after 300 km or 3 ...


The second query is the pattern in miniature: retrieval was weak, the
system *noticed* (grade), repaired its own input (rewrite), and retried —
all in explicit, debuggable control flow. No planner, no step budget, no
surprise costs.

## When to reach for which

| You know the steps? | Use |
|---|---|
| Yes, fixed | Pipeline (this notebook) |
| Yes, with checks and retries | Workflow + branches (this notebook) |
| No — genuinely open-ended | Agent loop (yesterday) |

The next lab gives these workflows the structure they deserve: a state
machine with nodes, edges, and conditional routing.